## Imports

In [ ]:
import random
import os

import numpy as np
import pandas as pd

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

## Utils

In [3]:
def get_data_root():
  '''
    Ritorna il percorso della cartella contenente i dati in base all'ambiente di esecuzione.
  '''
  try:
      import google.colab
      from google.colab import drive

      try:
          drive.mount("/content/drive", force_remount=True)
          return "/content/drive/MyDrive/ColabContent/Data_analytics"
      except Exception:
          print("Drive non montabile")
          return "/content"

  except ImportError:
      return "../../data"

print(get_data_root())

../../data


## Global variables

In [4]:
DATA_ROOT = get_data_root()
DATASET_PATH = f"{DATA_ROOT}/Dataset2526/train.csv"
TRAIN_SET_PATH = f"{DATA_ROOT}/train_processed.csv"
VAL_SET_PATH = f"{DATA_ROOT}/val_processed.csv"
TEST_SET_PATH = f"{DATA_ROOT}/test_processed.csv"
SEED = 42

## Seed per riproducibilità

In [ ]:
def fix_random(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True  # slower

## Impostazione GPU

In [ ]:
# look for GPU
if torch.backends.mps.is_available():
    print("MPS device is available.")
    device = torch.device("mps")
elif torch.cuda.is_available():
    print("CUDA device is available.")
    device = torch.device("cuda")
else:
    print("No GPU acceleration available.")
    device = torch.device("cpu")

## Definizione data loader

In [ ]:
class MyDataset(Dataset):
    def __init__(self, X, y):
        # Conversione esplicita in numpy array se necessario per evitare errori di indicizzazione
        self.X = torch.tensor(X.values if hasattr(X, 'values') else X, dtype=torch.float32)
        self.y = torch.tensor(y.values if hasattr(y, 'values') else y, dtype=torch.long)
        
        self.num_features = self.X.shape[1]
        self.num_classes = len(torch.unique(self.y))

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## Caricamento dati

In [5]:
train = pd.read_csv(TRAIN_SET_PATH)
val = pd.read_csv(VAL_SET_PATH)
test = pd.read_csv(TEST_SET_PATH)

X_train = train.drop(columns=['grade'])
y_train = train["grade"]

X_val = val.drop(columns=['grade'])
y_val = val["grade"]

X_test = test.drop(columns=['grade'])
y_test = test["grade"]

print(f"TRAIN:\nX: {X_train.shape}\ny: {y_train.shape}")
print(f"VAL:\nX: {X_val.shape}\ny: {y_val.shape}")
print(f"TEST:\nX: {X_test.shape}\ny: {y_test.shape}")

TRAIN:
X: (33747, 98)
y: (33747,)
VAL:
X: (14830, 98)
y: (14830,)
TEST:
X: (14831, 98)
y: (14831,)


# Modeling (Feed-Forward)

### Architettura feed-forward

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, input_size, num_classes, hidden_size):
        super(FeedForward, self).__init__()
        
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # Layer 1
        self.input_layer = nn.Linear(self.input_size, self.hidden_size)
        self.bn1 = nn.BatchNorm1d(self.hidden_size) 
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)            
        
        # Layer 2
        self.hidden_layer = nn.Linear(self.hidden_size, self.hidden_size // 2)
        self.relu2 = nn.ReLU()
        
        # Output
        self.output_layer = nn.Linear(self.hidden_size // 2, num_classes)
        

    def forward(self, x):
        h = self.input_layer(x)
        h = self.bn1(h)
        h = self.relu(h)
        h = self.dropout1(h)
        
        h = self.hidden_layer(h)
        h = self.relu2(h)
        
        output = self.output_layer(h)
        return output

### Funzione di train

In [ ]:
def train_model(model, criterion, optimizer, epoch, scheduler, train_loader, val_loader, device, writer, log_name="best_model"):
    n_iter = 0
    best_valid_loss = float('inf')
    for ep in range(epoch):
        model.train()
        
        for data, targets in train_loader:
            data, targets = data.to(device), targets.to(device)
            
            optimizer.zero_grad()

            # Forward pass
            y_pred = model(data)

            # Compute Loss
            loss = criterion(y_pred, targets)
            writer.add_scalar("Loss/train", loss, n_iter)

            # Backward pass
            loss.backward()
            optimizer.step()

            n_iter += 1
        
        labels, _, y_pred = test_model(model, val_loader, device)
        loss_val = criterion(y_pred, labels)
        writer.add_scalar("Loss/val", loss_val, ep)
        
        # save best model
        if loss_val.item() < best_valid_loss:
            best_valid_loss = loss_val.item()
            if not os.path.exists('models'):
                os.makedirs('models')
            torch.save(model.state_dict(), 'models/'+log_name)
        
        writer.add_scalar("Learning Rate", scheduler.get_last_lr()[0], ep)
        
        scheduler.step()
            
    return model

### Funzione di test

In [ ]:
# Define a function to evaluate the performance on validation and test sets

def test_model(model, data_loader, device):
    model.eval()
    y_pred = []
    y_test = []
    
    for data, targets in data_loader:
        data, targets = data.to(device), targets.to(device)
        y_pred += model(data)
        y_test += targets
    
    y_test = torch.stack(y_test).squeeze()
    y_pred = torch.stack(y_pred).squeeze()
    y_pred_c = y_pred.argmax(dim=1, keepdim=True).squeeze()
    
    return y_test, y_pred_c, y_pred

### Scaling

In [ ]:
from sklearn.preprocessing import RobustScaler

# Scaling
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Creazione Dataset
train_subset = MyDataset(X_train_scaled, y_train)
val_subset = MyDataset(X_val_scaled, y_val)
test_subset = MyDataset(X_test_scaled, y_test)

### Iperparametri

In [ ]:
# hyperparameters
num_epochs = 50
learning_rate = 0.001 # LR più basso per Adam
gamma = 0.5 # di quanto voglio aggiornare il learning rate ogni step
batch_size = 64        # Batch più grande per stabilità con BatchNorm
hidden_size = 128      # Più neuroni per gestire 98 feature
step_size = 20 

## Addestramento

In [ ]:
# fix the seed for reproducibility
fix_random(seed)


# Start tensorboard
writer = SummaryWriter(log_dir="runs/FeedForwardV1")

# Create relative dataloaders
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)


# Define the architecture, loss and optimizer
model = FeedForwardV1(train_subset.num_features, train_subset.num_classes, hidden_size)
model.to(device)
criterion = torch.nn.CrossEntropyLoss()
# optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)


# Test before the training
y_test, y_pred_c, _ = test_model(model, test_loader, device)
acc = (y_test == y_pred_c).float().sum() / y_test.shape[0]
print("Accuracy before training:", acc.cpu().numpy())


# Train the model 
model = train_model(model, criterion, optimizer, num_epochs, scheduler, train_loader, val_loader, device, writer)


# Load best model
model.load_state_dict(torch.load("models/best_model", weights_only=True))
model.to(device)


# Test after the training
y_test, y_pred_c, _ = test_model(model, test_loader, device)
acc = (y_test == y_pred_c).float().sum() / y_test.shape[0]
print("Accuracy after training:", acc.cpu().numpy())


# Close tensorboard writer after a training
writer.flush()
writer.close()